In [ ]:
import os
import time
import copy
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# =============================================================================
# 0. Unified Configuration
# =============================================================================
CFG = {
    # reproducibility / device / dtype
    "seed": 0,
    "seeds": [0, 1, 2, 3, 4],
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "dtype": torch.float64,

    # problem params
    "alpha": 0.1,
    "beta": 0.2,
    "gamma": 0.5,
    "k_terminal": 1.0,

    # fixed maturity setting
    "T": 1.0,
    "N": 20,              # If None, set N = round(10*T), i.e., h = 0.1
    "delta": 0.4,
    "use_delta_equal_03T": False,

    "X0": 1.0,

    # training params
    "train_paths": 256,
    "hidden": 64,
    "lr": 3e-4,

    # iterations
    "n_iters": 2000,
    "fabsde_iters": 10000,

    # PGDPO projection
    "proj_rollouts": 8192,

    # logging
    "pg_print_every": 200,
    "fabsde_print_every": 500,

    # output
    "outdir": "figures_pdf_fixed_T_multiseed",
}


# =============================================================================
# 1. Matplotlib Style
# =============================================================================
plt.rcParams.update({
    "pdf.fonttype": 42,
    "font.family": "serif",
    "font.serif": ["Liberation Serif", "FreeSerif", "serif"],
    "font.size": 10,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "mathtext.fontset": "stix",
})


# =============================================================================
# 2. Config Finalization / Utilities
# =============================================================================
def finalize_cfg(cfg: dict) -> dict:
    cfg = copy.deepcopy(cfg)

    cfg["T"] = float(cfg["T"])

    if cfg["N"] is None:
        cfg["N"] = int(round(10 * cfg["T"]))
    else:
        cfg["N"] = int(cfg["N"])

    if cfg["N"] <= 0:
        raise ValueError("CFG['N'] must be positive.")

    if cfg["use_delta_equal_03T"]:
        cfg["delta"] = 0.3 * cfg["T"]

    cfg["delta"] = float(cfg["delta"])

    os.makedirs(cfg["outdir"], exist_ok=True)

    return cfg


def cfg_for_seed(cfg: dict, seed: int) -> dict:
    out = copy.deepcopy(cfg)
    out["seed"] = int(seed)
    return out


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device_and_dtype(cfg: dict):
    device = torch.device(cfg["device"])
    dtype = cfg["dtype"]
    torch.set_default_dtype(dtype)
    return device, dtype


# =============================================================================
# 3. Physics
# =============================================================================
class Physics:
    def __init__(self, cfg: dict, device: torch.device):
        self.alpha = float(cfg["alpha"])
        self.beta = float(cfg["beta"])
        self.gamma = float(cfg["gamma"])
        self.k_term = float(cfg["k_terminal"])

        self.T = float(cfg["T"])
        self.N = int(cfg["N"])
        self.h = self.T / self.N

        self.delta = float(cfg["delta"])
        self.delay_steps = int(round(self.delta / self.h))
        self.device = device

    def step(self, x_now, x_lag, u, dW):
        drift = self.alpha * x_lag - u
        diff = self.beta * x_lag
        return x_now + drift * self.h + diff * dW

    def utility(self, u):
        return (u ** self.gamma) / self.gamma

    def get_terminal_reward(self, x_final):
        return self.k_term * x_final

    def init_history(self, batch_size: int, x0: float):
        return [
            torch.full((batch_size,), x0, device=self.device)
            for _ in range(self.delay_steps + 1)
        ]


# =============================================================================
# 4. Models
# =============================================================================
class BaseLSTM(nn.Module):
    def __init__(self, output_dim: int, hidden_dim: int):
        super().__init__()
        self.lstm = nn.LSTM(input_size=3, hidden_size=hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, t, x, x_lag, hidden):
        inp = torch.stack([t, x, x_lag], dim=1).unsqueeze(1)
        out, new_hidden = self.lstm(inp, hidden)
        return self.fc(out[:, -1, :]), new_hidden


class PolicyNet(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.lstm = nn.LSTM(input_size=2, hidden_size=hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, t, x, hidden):
        inp = torch.stack([t, x], dim=1).unsqueeze(1)  # (B, 1, 2)
        out, h_new = self.lstm(inp, hidden)
        raw = self.fc(out[:, -1, :])
        c = torch.nn.functional.softplus(raw).squeeze(-1)
        return c, h_new


class FABSDE_Net(BaseLSTM):
    def __init__(self, hidden_dim: int):
        super().__init__(output_dim=4, hidden_dim=hidden_dim)

    def forward(self, t, x, x_lag, hidden):
        raw, new_hidden = super().forward(t, x, x_lag, hidden)

        Y = torch.nn.functional.softplus(raw[:, 0]) + 1e-6
        Z = raw[:, 1]
        EY = torch.nn.functional.softplus(raw[:, 2]) + 1e-6
        EZ = raw[:, 3]

        return Y, Z, EY, EZ, new_hidden


# =============================================================================
# 5. Analytic Benchmark
# =============================================================================
def solve_analytic(cfg: dict):
    phy = Physics(cfg, device=torch.device(cfg["device"]))
    N = int(cfg["N"])
    p = np.zeros(N + 1, dtype=np.float64)

    start_idx = max(N - phy.delay_steps, 0)
    p[start_idx:] = cfg["k_terminal"]

    for i in range(start_idx - 1, -1, -1):
        future_idx = i + 1 + phy.delay_steps
        if future_idx <= N:
            p[i] = p[i + 1] + cfg["alpha"] * phy.h * p[future_idx]
        else:
            p[i] = p[i + 1]

    c_star = np.maximum(p, 1e-12) ** (1.0 / (cfg["gamma"] - 1.0))
    return c_star, p


# =============================================================================
# 6. Stage I: PG / LSTM-DPO
# =============================================================================
def train_pg(policy: nn.Module, phy: Physics, cfg: dict):
    print(">>> Training PG / LSTM-DPO...")
    opt = optim.Adam(policy.parameters(), lr=cfg["lr"])

    for it in range(1, cfg["n_iters"] + 1):
        opt.zero_grad()

        dW = torch.randn(
            cfg["train_paths"],
            cfg["N"],
            device=phy.device
        ) * np.sqrt(phy.h)

        X = torch.zeros(
            cfg["train_paths"],
            cfg["N"] + phy.delay_steps + 1,
            device=phy.device
        )
        X[:, : phy.delay_steps + 1] = cfg["X0"]

        run_util = 0.0
        hidden = None

        for n in range(cfg["N"]):
            k = phy.delay_steps + n
            t_val = torch.full((cfg["train_paths"],), n * phy.h, device=phy.device)

            x_now = X[:, k]
            x_lag = X[:, n]

            # Policy receives only current state x_now.
            c_n, hidden = policy(t_val, x_now, hidden)

            # Dynamics still uses x_lag.
            X[:, k + 1] = phy.step(x_now, x_lag, c_n, dW[:, n])

            run_util = run_util + phy.utility(c_n) * phy.h

        loss = -(run_util + phy.get_terminal_reward(X[:, -1])).mean()

        if it == 1 or it % cfg["pg_print_every"] == 0 or it == cfg["n_iters"]:
            print(f"  iter {it:5d} | LSTM-DPO loss = {loss.item():.10f}")

        loss.backward()
        opt.step()

    return policy


@torch.no_grad()
def eval_pg_policy(policy: nn.Module, phy: Physics, cfg: dict):
    c_pg = []
    hidden = None
    X = phy.init_history(1, cfg["X0"])

    for i in range(cfg["N"]):
        k = phy.delay_steps + i
        t_i = torch.tensor([i * phy.h], device=phy.device)

        x_now = X[k]
        x_lag = X[i]

        # Policy receives only current state x_now.
        c_i, hidden = policy(t_i, x_now, hidden)
        c_pg.append(float(c_i.item()))

        # Dynamics still uses x_lag.
        x_next = phy.step(x_now, x_lag, c_i, 0.0)
        X.append(x_next)

    return np.array(c_pg, dtype=np.float64)

def sample_antithetic_dW(batch_size: int, h: float, device: torch.device):
    half = batch_size // 2
    scale = np.sqrt(h)

    eps_half = torch.randn(half, device=device) * scale
    dW = torch.cat([eps_half, -eps_half], dim=0)

    if batch_size % 2 == 1:
        extra = torch.randn(1, device=device) * scale
        dW = torch.cat([dW, extra], dim=0)

    return dW

# =============================================================================
# 7. Stage II: PGDPO Projection
# =============================================================================
def compute_pgdpo_with_timing(base_policy: nn.Module, phy: Physics, cfg: dict):
    print(">>> Computing PGDPO projection...")

    c_proj = []
    p_proj = []
    step_times = []

    real_hist = [
        torch.tensor([cfg["X0"]], device=phy.device)
        for _ in range(phy.delay_steps + 1)
    ]

    real_hidden_hist = [None]

    def _clone_hidden(hidden):
        if hidden is None:
            return None
        h, c = hidden
        return h.detach().clone(), c.detach().clone()

    def _expand_hidden(hidden, batch_size: int):
        if hidden is None:
            return None
        h, c = hidden
        h = h.expand(-1, batch_size, -1).contiguous()
        c = c.expand(-1, batch_size, -1).contiguous()
        return h, c

    hidden_real = None
    total_t0 = time.time()

    for i in range(cfg["N"]):
        step_t0 = time.time()

        x_start = real_hist[-1].detach().clone().requires_grad_(True)
        n_inner = int(cfg["proj_rollouts"])

        hidden_prefix = real_hidden_hist[i]
        hidden_inner = _expand_hidden(hidden_prefix, n_inner)

        if phy.delay_steps > 0:
            past_hist = torch.stack(real_hist[:-1]).detach()
            past_hist = past_hist[-phy.delay_steps:]
            past_hist = past_hist.expand(-1, n_inner)

            curr_state = x_start.expand(n_inner)
            inner_X = [past_hist[k] for k in range(phy.delay_steps)] + [curr_state]
        else:
            inner_X = [x_start.expand(n_inner)]

        J_future = 0.0

        for step in range(cfg["N"] - i):
            abs_step = i + step
            k_idx = phy.delay_steps + step

            t_val = torch.full((n_inner,), abs_step * phy.h, device=phy.device)

            x_now = inner_X[k_idx]
            x_lag = inner_X[step]

            # Policy receives only current state x_now.
            c_val, hidden_inner = base_policy(t_val, x_now, hidden_inner)

            # Dynamics still uses x_lag, with antithetic Brownian increments.
            dW = sample_antithetic_dW(n_inner, phy.h, device=phy.device)
            x_next = phy.step(x_now, x_lag, c_val, dW)

            inner_X.append(x_next)
            J_future = J_future + phy.utility(c_val) * phy.h

        J_total = J_future + phy.get_terminal_reward(inner_X[-1])

        grad_x = torch.autograd.grad(
            J_total.sum(),
            x_start,
            retain_graph=False,
            create_graph=False
        )[0]

        p_t = float(grad_x.item()) / n_inner

        p_clamped = max(p_t, 1e-12)
        c_star = float(p_clamped ** (1.0 / (cfg["gamma"] - 1.0)))

        c_proj.append(c_star)
        p_proj.append(p_t)

        if phy.delay_steps > 0:
            x_lag_real = real_hist[-(phy.delay_steps + 1)]
        else:
            x_lag_real = real_hist[-1]

        t_real = torch.tensor([i * phy.h], device=phy.device)

        with torch.no_grad():
            # Real hidden state also receives only current state.
            _, hidden_real = base_policy(
                t_real,
                real_hist[-1],
                hidden_real
            )

        x_next_real = phy.step(
            real_hist[-1],
            x_lag_real,
            torch.tensor([c_star], device=phy.device),
            0.0
        )

        real_hist.append(x_next_real.detach())
        real_hidden_hist.append(_clone_hidden(hidden_real))

        step_times.append(time.time() - step_t0)

    total_elapsed = time.time() - total_t0
    avg_step_elapsed = float(np.mean(step_times)) if len(step_times) > 0 else np.nan

    return (
        np.array(c_proj, dtype=np.float64),
        np.array(p_proj, dtype=np.float64),
        float(total_elapsed),
        float(avg_step_elapsed),
        np.array(step_times, dtype=np.float64),
    )


# =============================================================================
# 8. FABSDE
# =============================================================================
def train_fabsde(phy: Physics, cfg: dict):
    print(">>> Training FABSDE...")
    net = FABSDE_Net(hidden_dim=cfg["hidden"]).to(phy.device)
    opt = optim.Adam(net.parameters(), lr=cfg["lr"])

    for it in range(1, cfg["fabsde_iters"] + 1):
        opt.zero_grad()

        B = cfg["train_paths"]
        dW = torch.randn(B, cfg["N"], device=phy.device) * np.sqrt(phy.h)
        X = phy.init_history(B, cfg["X0"])

        preds = {"Y": [], "Z": [], "EY": [], "EZ": []}
        tildeY = [None] * (cfg["N"] + 1)
        hidden = None

        for i in range(cfg["N"]):
            k = phy.delay_steps + i
            t_i = torch.full((B,), i * phy.h, device=phy.device)

            Y, Z, EY, EZ, hidden = net(t_i, X[k], X[i], hidden)

            preds["Y"].append(Y)
            preds["Z"].append(Z)
            preds["EY"].append(EY)
            preds["EZ"].append(EZ)

            c_i = Y.pow(1.0 / (cfg["gamma"] - 1.0))
            X.append(phy.step(X[k], X[i], c_i, dW[:, i]))

            if i <= cfg["N"] - phy.delay_steps:
                f_i = cfg["alpha"] * EY + cfg["beta"] * EZ
            else:
                f_i = torch.zeros_like(Y)

            tildeY[i + 1] = Y - f_i * phy.h + Z * dW[:, i]

        t_N = torch.full((B,), cfg["N"] * phy.h, device=phy.device)
        Y_N, Z_N, EY_N, EZ_N, _ = net(t_N, X[-1], X[cfg["N"]], hidden)

        preds["Y"].append(Y_N)
        preds["Z"].append(Z_N)
        preds["EY"].append(EY_N)
        preds["EZ"].append(EZ_N)

        L1 = sum(
            (preds["Y"][i + 1] - tildeY[i + 1]).pow(2).mean()
            for i in range(cfg["N"])
        )
        L1 = L1 + (preds["Y"][cfg["N"]] - cfg["k_terminal"]).pow(2).mean()

        L2 = 0.0
        if phy.delay_steps <= cfg["N"]:
            limit = cfg["N"] - phy.delay_steps

            L2 = L2 + sum(
                (preds["EY"][i] - tildeY[i + phy.delay_steps]).pow(2).mean()
                for i in range(limit + 1)
            )

            L2 = L2 + sum(
                (preds["EZ"][i] - preds["Z"][i + phy.delay_steps]).pow(2).mean()
                for i in range(limit + 1)
            )

        loss = L1 + L2

        loss.backward()
        opt.step()

        if it == 1 or it % cfg["fabsde_print_every"] == 0 or it == cfg["fabsde_iters"]:
            print(f"  iter {it:5d} | FABSDE loss = {loss.item():.10f}")

    with torch.no_grad():
        c_path = []
        Y_path = []
        hidden = None
        X = phy.init_history(1, cfg["X0"])

        for i in range(cfg["N"]):
            k = phy.delay_steps + i
            t_val = torch.tensor([i * phy.h], device=phy.device)

            res = net(t_val, X[k], X[i], hidden)
            hidden = res[-1]

            c_val = res[0].pow(1.0 / (cfg["gamma"] - 1.0))

            c_path.append(c_val.item())
            Y_path.append(res[0].item())

            X.append(phy.step(X[k], X[i], c_val, 0.0))

    return np.array(c_path, dtype=np.float64), np.array(Y_path, dtype=np.float64)


# =============================================================================
# 9. Metrics / Plotting
# =============================================================================
def compute_rmse_mae(gt: np.ndarray, pred: np.ndarray):
    L = min(len(gt), len(pred))
    err = pred[:L] - gt[:L]

    rmse = float(np.sqrt(np.mean(err ** 2)))
    mae = float(np.mean(np.abs(err)))

    return rmse, mae


def mean_std(values):
    values = np.asarray(values, dtype=np.float64)
    mean = float(np.mean(values))
    std = float(np.std(values, ddof=1)) if len(values) > 1 else 0.0
    return mean, std


def print_seed_metrics_table(results):
    print("\n" + "#" * 90)
    print("Seed-wise Metrics")
    print("#" * 90)

    header = (
        f"{'seed':>5} | {'Algorithm':<10} | "
        f"{'RMSE':>12} | {'MAE':>12}"
    )
    print(header)
    print("-" * len(header))

    for r in results:
        seed = r["seed"]
        rows = [
            ("PG", r["pg_rmse"], r["pg_mae"]),
            ("PGDPO", r["pgdpo_rmse"], r["pgdpo_mae"]),
            ("FABSDE", r["fabsde_rmse"], r["fabsde_mae"]),
        ]

        for name, rmse, mae in rows:
            print(
                f"{seed:5d} | {name:<10} | "
                f"{rmse:12.6e} | {mae:12.6e}"
            )


def print_multiseed_summary(results):
    print("\n" + "#" * 90)
    print("Multi-seed Summary: mean ± std over seeds")
    print("#" * 90)

    header = (
        f"{'Algorithm':<10} | "
        f"{'RMSE mean':>12} | {'RMSE std':>12} | "
        f"{'MAE mean':>12} | {'MAE std':>12}"
    )
    print(header)
    print("-" * len(header))

    algo_map = [
        ("PG", "pg"),
        ("PGDPO", "pgdpo"),
        ("FABSDE", "fabsde"),
    ]

    for algo_name, key in algo_map:
        rmse_vals = [r[f"{key}_rmse"] for r in results]
        mae_vals = [r[f"{key}_mae"] for r in results]

        rmse_mean, rmse_std = mean_std(rmse_vals)
        mae_mean, mae_std = mean_std(mae_vals)

        print(
            f"{algo_name:<10} | "
            f"{rmse_mean:12.6e} | {rmse_std:12.6e} | "
            f"{mae_mean:12.6e} | {mae_std:12.6e}"
        )


def curve_mean_std(curves):
    arr = np.asarray(curves, dtype=np.float64)
    mean = np.mean(arr, axis=0)
    std = np.std(arr, axis=0, ddof=1) if arr.shape[0] > 1 else np.zeros_like(mean)
    return mean, std


def plot_controls_with_bands(
    t_grid: np.ndarray,
    gt: np.ndarray,
    curves_by_algo: dict,
    save_path: str = None,
):
    fig, ax = plt.subplots(figsize=(7, 5))

    ax.plot(t_grid, gt, "k-", lw=2.2, label="Benchmark")

    style = {
        "PG": {
            "color": "tab:blue",
            "mean_ls": ":",
            "label": "LSTM-DPO",
        },
        "FABSDE": {
            "color": "tab:green",
            "mean_ls": "-.",
            "label": "DEEP ABSDE",
        },
        "PGDPO": {
            "color": "tab:red",
            "mean_ls": "--",
            "label": "PGDPO",
        },
    }

    for algo in ["PG", "FABSDE", "PGDPO"]:
        mean_curve, std_curve = curve_mean_std(curves_by_algo[algo])
        upper = mean_curve + std_curve
        lower = mean_curve - std_curve

        color = style[algo]["color"]

        ax.plot(
            t_grid,
            mean_curve,
            linestyle=style[algo]["mean_ls"],
            color=color,
            lw=2.0,
            label=style[algo]["label"],
        )

        ax.fill_between(
            t_grid,
            lower,
            upper,
            color=color,
            alpha=0.16,
            linewidth=0.0,
        )

        # band upper/lower boundaries as dotted lines
        ax.plot(t_grid, upper, linestyle=":", color=color, lw=1.25, alpha=0.95)
        ax.plot(t_grid, lower, linestyle=":", color=color, lw=1.25, alpha=0.95)

    ax.set_xlabel("Time")
    ax.set_ylabel("Consumption")
    ax.legend()
    ax.grid(alpha=0.1)

    fig.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, format="pdf", bbox_inches="tight")
        print(f"[Saved] {save_path}")

    plt.show()
    plt.close(fig)


# =============================================================================
# 10. One Fixed-T Run for One Seed
# =============================================================================
def run_fixed_T_single_seed(cfg: dict):
    cfg = finalize_cfg(cfg)

    set_seed(cfg["seed"])
    device, _ = get_device_and_dtype(cfg)
    phy = Physics(cfg, device=device)

    print("\n" + "=" * 110)
    print(
        f"[RUN] seed={cfg['seed']} | Fixed T={cfg['T']}, N={cfg['N']}, h={phy.h:.6f}, "
        f"delta={cfg['delta']}, delay_steps={phy.delay_steps}, device={device}"
    )
    print("=" * 110)

    # 1) Analytic benchmark
    c_ana, p_ana = solve_analytic(cfg)
    gt = c_ana[:-1]

    # 2) Stage I: PG / LSTM-DPO
    t0_pg = time.time()

    policy_pg = PolicyNet(hidden_dim=cfg["hidden"]).to(device)
    policy_pg = train_pg(policy_pg, phy, cfg)

    pg_train_time = time.time() - t0_pg
    c_pg = eval_pg_policy(policy_pg, phy, cfg)

    # 3) Stage II: PGDPO
    c_proj, p_proj, proj_total_time, proj_avg_step_time, proj_step_times = compute_pgdpo_with_timing(
        policy_pg,
        phy,
        cfg
    )

    # 4) FABSDE
    t0_fabsde = time.time()

    c_fabsde, y_fabsde = train_fabsde(phy, cfg)

    fabsde_train_time = time.time() - t0_fabsde

    # 5) Metrics
    pg_rmse, pg_mae = compute_rmse_mae(gt, c_pg)
    pgdpo_rmse, pgdpo_mae = compute_rmse_mae(gt, c_proj)
    fabsde_rmse, fabsde_mae = compute_rmse_mae(gt, c_fabsde)

    print("\n[Metrics]")
    print(f"{'Algorithm':<15} | {'RMSE':<12} | {'MAE':<12}")
    print("-" * 46)
    print(f"{'PG':<15} | {pg_rmse:<12.8f} | {pg_mae:<12.8f}")
    print(f"{'PGDPO':<15} | {pgdpo_rmse:<12.8f} | {pgdpo_mae:<12.8f}")
    print(f"{'FABSDE':<15} | {fabsde_rmse:<12.8f} | {fabsde_mae:<12.8f}")

    print("\n[Timing]")
    print(f"PG training time (sec)            : {pg_train_time:.6f}")
    print(f"PGDPO projection total time (sec) : {proj_total_time:.6f}")
    print(f"PGDPO projection avg/step (sec)   : {proj_avg_step_time:.6f}")
    print(f"PGDPO projection min/step (sec)   : {proj_step_times.min():.6f}")
    print(f"PGDPO projection max/step (sec)   : {proj_step_times.max():.6f}")
    print(f"FABSDE training time (sec)        : {fabsde_train_time:.6f}")

    result = {
        "cfg": cfg,
        "seed": int(cfg["seed"]),
        "T": cfg["T"],
        "N": cfg["N"],
        "h": phy.h,
        "delta": cfg["delta"],
        "delay_steps": phy.delay_steps,

        "gt": gt,
        "p_ana": p_ana,
        "c_pg": c_pg,
        "c_proj": c_proj,
        "c_fabsde": c_fabsde,
        "p_proj": p_proj,
        "y_fabsde": y_fabsde,

        "pg_train_time_sec": float(pg_train_time),
        "pgdpo_total_projection_sec": float(proj_total_time),
        "pgdpo_avg_step_sec": float(proj_avg_step_time),
        "pgdpo_min_step_sec": float(np.min(proj_step_times)),
        "pgdpo_max_step_sec": float(np.max(proj_step_times)),
        "fabsde_train_time_sec": float(fabsde_train_time),

        "pg_rmse": pg_rmse,
        "pg_mae": pg_mae,
        "pgdpo_rmse": pgdpo_rmse,
        "pgdpo_mae": pgdpo_mae,
        "fabsde_rmse": fabsde_rmse,
        "fabsde_mae": fabsde_mae,

        "proj_step_times": proj_step_times,
    }

    del policy_pg
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


# =============================================================================
# 11. Multi-seed Fixed-T Experiment
# =============================================================================
def run_fixed_T_multiseed_experiment(cfg: dict):
    cfg = finalize_cfg(cfg)
    seeds = list(cfg["seeds"])

    all_results = []

    for seed in seeds:
        run_cfg = cfg_for_seed(cfg, seed)
        result = run_fixed_T_single_seed(run_cfg)
        all_results.append(result)

    print_seed_metrics_table(all_results)
    print_multiseed_summary(all_results)

    # Plot mean ± std bands
    ref = all_results[0]
    t_grid = np.linspace(0.0, ref["T"], ref["N"])

    curves_by_algo = {
        "PG": np.stack([r["c_pg"] for r in all_results], axis=0),
        "PGDPO": np.stack([r["c_proj"] for r in all_results], axis=0),
        "FABSDE": np.stack([r["c_fabsde"] for r in all_results], axis=0),
    }

    fig_path = os.path.join(
        cfg["outdir"],
        (
            f"controls_fixed_T{ref['T']:.1f}_N{ref['N']}"
            f"_delta{ref['delta']:.2f}_seeds{seeds[0]}to{seeds[-1]}_band.pdf"
        )
    )

    plot_controls_with_bands(
        t_grid=t_grid,
        gt=ref["gt"],
        curves_by_algo=curves_by_algo,
        save_path=None,#fig_path,
    )

    # Zip output PDFs
    zip_path = os.path.join(
        cfg["outdir"],
        f"figures_pdf_fixed_T{ref['T']:.1f}_multiseed.zip"
    )

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for fname in os.listdir(cfg["outdir"]):
            if fname.endswith(".pdf"):
                full_path = os.path.join(cfg["outdir"], fname)
                zipf.write(full_path, arcname=fname)

    #print(f"[ZIP saved] {zip_path}")

    output = {
        "cfg": cfg,
        "results": all_results,
        "fig_path": fig_path,
        "zip_path": zip_path,
    }

    return output


# =============================================================================
# 12. Execute
# =============================================================================
RESULT = run_fixed_T_multiseed_experiment(CFG)